# **Exercícios Aula 2**: Engenharia e Qualidade de Dados

----

- Você recebeu um arquivo extraído de um sistema legado com 100.000 registros. 

- O dado está "sujo", com problemas de encoding, duplicatas, valores nulos e informações sensíveis que violam a LGPD.

- Sua missão: Transformar esse "pântano de dados" (Data Swamp) em um conjunto pronto para análise (Data Lake).

- O .csv pode ser adquirido via: https://drive.google.com/file/d/1YLUSKsgzhU-7HAbPHiVbONPdATHw1UJP/view?usp=sharing

-----------------

## EXERCÍCIO 1: Diagnóstico e Qualidade
1. Carregue o arquivo e exiba o info() do DataFrame.

2. Quantas linhas duplicadas existem no total?

3. Conte quantas vezes a string "ERRO" aparece e quantos valores estão nulos (NaN).

4. Qual a memória RAM consumida atualmente por este DataFrame?

5. Por que a coluna valor_total está consumindo tanta memória e por que o tipo é object?

In [0]:
import pandas as pd
import numpy as np

# 1. Carregar o arquivo
df = pd.read_csv('./dados_aula02_atividades_vendas_brutas.csv')

print("=== INFO DO DATAFRAME ===")
print(df.info())
print("\n" + "="*50)

# 2. Linhas duplicadas
qtd_duplicadas = df.duplicated().sum()
print(f"\n2. Linhas duplicadas: {qtd_duplicadas}")

# 3. Contar "ERRO" e NaN
erros_count = df.astype(str).apply(lambda x: x.str.contains('ERRO', case=False, na=False)).sum().sum()
nulos_count = df.isnull().sum().sum()
print(f"\n3. Quantidade de 'ERRO': {erros_count}")
print(f"   Valores nulos (NaN): {nulos_count}")

# 4. Memória RAM consumida
memoria_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"\n4. Memória RAM consumida: {memoria_mb:.2f} MB")
print("\n=== Memória por coluna ===")
print(df.memory_usage(deep=True) / 1024**2)

# 5. Análise da coluna valor_total
print(f"\n5. Tipo da coluna 'valor_total': {df['valor_total'].dtype}")
print(f"   A coluna está como 'object' porque contém texto ('R$', vírgulas)")
print(f"   e valores mistos (números + 'ERRO' + NaN), impedindo conversão para numérico.")
print(f"   Isso consome mais memória que um tipo float64.")
print("\n   Exemplo de valores:")
print(df['valor_total'].head(10))

---

## EXERCÍCIO 2: Limpeza e Integridade
1. Remova todas as linhas duplicadas.

2. Trate os valores nulos da coluna valor_total: substitua-os pelo valor "R$ 0,00".

3. Remova definitivamente as linhas que contenham o texto "ERRO" na coluna de valores.

4. Verifique novamente se ainda existem nulos ou a palavra "ERRO".

---

In [0]:
print("=== LIMPEZA E INTEGRIDADE ===")
print(f"Linhas antes da limpeza: {len(df)}\n")

# 1. Remover duplicadas
df = df.drop_duplicates()
print(f"1. Após remover duplicadas: {len(df)} linhas")

# 2. Substituir nulos na coluna valor_total por "R$ 0,00"
df['valor_total'] = df['valor_total'].fillna('R$ 0,00')
print(f"2. Nulos substituídos por 'R$ 0,00'")

# 3. Remover linhas com "ERRO" na coluna valor_total
antes_erro = len(df)
df = df[~df['valor_total'].astype(str).str.contains('ERRO', case=False, na=False)]
removidas = antes_erro - len(df)
print(f"3. Removidas {removidas} linhas contendo 'ERRO'")
print(f"   Total de linhas agora: {len(df)}")

# 4. Verificar se ainda existem nulos ou "ERRO"
print("\n4. Verificação final:")
nulos_final = df.isnull().sum().sum()
erros_final = df.astype(str).apply(lambda x: x.str.contains('ERRO', case=False, na=False)).sum().sum()
print(f"   Valores nulos restantes: {nulos_final}")
print(f"   Valores 'ERRO' restantes: {erros_final}")

if nulos_final == 0 and erros_final == 0:
    print("   ✓ Dados limpos! Sem nulos e sem ERRO.")
else:
    print("   ✗ Ainda existem problemas nos dados.")

## EXERCÍCIO 3: Transformação e Vetorização
1. A coluna valor_total é do tipo object (texto).

2. Crie uma função ou use .str.replace para converter essa coluna para float.

3. Vetorização: Crie uma nova coluna chamada valor_com_desconto, que deve ser o valor_total menos 10% (apenas para pedidos com status "Pago").

---

In [0]:
print("=== TRANSFORMAÇÃO E VETORIZAÇÃO ===")

# 1 e 2. Converter valor_total de object para float
print(f"Tipo original: {df['valor_total'].dtype}")
print("\nExemplo antes da conversão:")
print(df['valor_total'].head())

# Remover 'R$', espaços e substituir vírgula por ponto
df['valor_total'] = df['valor_total'].str.replace('R$', '', regex=False)
df['valor_total'] = df['valor_total'].str.replace(' ', '')
df['valor_total'] = df['valor_total'].str.replace(',', '.')
df['valor_total'] = df['valor_total'].astype(float)

print(f"\nTipo após conversão: {df['valor_total'].dtype}")
print("\nExemplo após conversão:")
print(df['valor_total'].head())

# 3. Vetorização: criar coluna valor_com_desconto (10% desconto para status "Pago")
df['valor_com_desconto'] = df.apply(
    lambda row: row['valor_total'] * 0.9 if row['status'] == 'Pago' else row['valor_total'],
    axis=1
)

print("\n=== COLUNA valor_com_desconto CRIADA ===")
print("\nExemplos (primeiras 10 linhas):")
print(df[['status', 'valor_total', 'valor_com_desconto']].head(10))

# Estatísticas
total_pagos = (df['status'] == 'Pago').sum()
economia_total = (df[df['status'] == 'Pago']['valor_total'] * 0.1).sum()
print(f"\n✓ {total_pagos} pedidos pagos receberam 10% de desconto")
print(f"✓ Economia total para clientes: R$ {economia_total:,.2f}")

## EXERCÍCIO 4: Segurança de Dados (LGPD)

1. A coluna cliente_email é um PII (Personally Identifiable Information). Use a biblioteca hashlib ou uma técnica de substituição para anonimizar os e-mails dos clientes.

2. Exiba as 5 primeiras linhas para verificar se o e-mail agora está criptografado/ilegível.

---

In [0]:
import hashlib

print("=== SEGURANÇA DE DADOS (LGPD) ===")

print("\nAntes da anonimização:")
print(df[['cliente_email']].head())

# 1. Anonimizar e-mails usando SHA256
def anonimizar_email(email):
    """Criptografa o e-mail usando SHA256"""
    if pd.isna(email):
        return None
    return hashlib.sha256(str(email).encode()).hexdigest()

# Aplicar anonimização
df['cliente_email'] = df['cliente_email'].apply(anonimizar_email)

print("\n✓ E-mails anonimizados com sucesso!")

# 2. Exibir 5 primeiras linhas
print("\nApós anonimização (5 primeiras linhas):")
print(df[['cliente_email']].head())

print("\n✓ Os e-mails agora são hashes irreversíveis (SHA256)")
print("✓ Conformidade com LGPD: dados pessoais identificadores foram anonimizados")
print("\n=== EXEMPLO DE DADOS PROTEGIDOS ===")
print(df.head())

## EXERCÍCIO 5: Otimização de I/O
1. Salve o seu DataFrame limpo em dois formatos: vendas_limpas.csv e vendas_limpas.parquet.

2. Use a biblioteca os para verificar o tamanho (em KB) de ambos os arquivos no disco.

3. Ler apenas a coluna valor_total do arquivo Parquet.

---

In [0]:
import os

print("=== OTIMIZAÇÃO DE I/O ===")

# 1. Salvar em CSV e Parquet
print("\n1. Salvando DataFrame limpo...")
df.to_csv('vendas_limpas.csv', index=False)
df.to_parquet('vendas_limpas.parquet', index=False)
print("✓ Arquivos salvos: vendas_limpas.csv e vendas_limpas.parquet")

# 2. Verificar tamanho dos arquivos
print("\n2. Comparando tamanhos dos arquivos:")
tamanho_csv = os.path.getsize('vendas_limpas.csv') / 1024  # KB
tamanho_parquet = os.path.getsize('vendas_limpas.parquet') / 1024  # KB

print(f"   CSV:     {tamanho_csv:,.2f} KB")
print(f"   Parquet: {tamanho_parquet:,.2f} KB")
print(f"\n   ✓ Parquet é {tamanho_csv/tamanho_parquet:.1f}x menor que CSV!")
print(f"   ✓ Economia de espaço: {((tamanho_csv - tamanho_parquet) / tamanho_csv * 100):.1f}%")

# 3. Ler apenas coluna específica do Parquet
print("\n3. Lendo apenas a coluna 'valor_total' do Parquet:")
df_valor_only = pd.read_parquet('vendas_limpas.parquet', columns=['valor_total'])
print(df_valor_only.head(10))
print(f"\n✓ Leitura seletiva: apenas {len(df_valor_only.columns)} coluna carregada")
print(f"✓ Economia de memória ao ler apenas colunas necessárias")

# Estatísticas finais
print("\n" + "="*50)
print("=== RESUMO FINAL ===")
print(f"✓ DataFrame limpo: {len(df)} linhas")
print(f"✓ Colunas: {list(df.columns)}")
print(f"✓ Memória otimizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"✓ Dados salvos em CSV e Parquet")
print(f"✓ E-mails anonimizados (LGPD)")
print("="*50)